# Filtrado y selección de filas



**Fase 1 · Sesión 2** — Dataset: Palmer Penguins

Cómo quedarme con las filas que cumplen una o varias condiciones.
Es el 80 % del trabajo diario con datos.

## Chuleta: los operadores de Pandas

Python decide **una** cosa. Pandas necesita decidir **una por fila**, porque
filtrar es marcar qué fila se queda y cuál se va.

| Python | Pandas | Significa |
|---|---|---|
| `and` | `&` | las dos condiciones |
| `or` | `\|` | una u otra |
| `not` | `~` | lo contrario |
| `in` | `.isin([...])` | está en esta lista |

**`and` no falla por capricho.** Recibe 344 valores y tiene que devolver uno solo:
no sabe cuál elegir, así que lanza `ValueError: the truth value is ambiguous`.
`&` no resume nada, compara fila a fila y devuelve otros 344.

**Los paréntesis son obligatorios.** `&` tiene más prioridad que `==` y `>`, así que
sin ellos Python lee `"Adelie" & df[...]` y peta con un error que no orienta nada.

In [28]:
import pandas as pd
import seaborn as sns

df = sns.load_dataset("penguins")

## Los dos pasos de un filtro

Esto es lo que más confunde al principio, así que queda escrito:

1. **La condición fabrica una máscara**: 344 True/False, uno por fila. No filtra nada.
2. **Los corchetes la aplican**: `df[mascara]` devuelve las filas marcadas con True.

Es como una plantilla perforada sobre la tabla: donde hay True hay agujero.

In [29]:
mascara_adelies = (df["species"] == "Adelie") & (df["body_mass_g"] > 4000)

print(mascara_adelies.head())   # los BOOLEANOS: la máscara en bruto
print(mascara_adelies.sum())    # cuántos cumplen (True vale 1, False vale 0)
print(df[mascara_adelies].shape)  # las FILAS de verdad: 35 pingüinos

# Guardar la máscara en una variable no es solo estético: si la vas a usar más de
# una vez, evita repetir una línea de 90 caracteres donde es fácil colar un fallo.

0    False
1    False
2    False
3    False
4    False
dtype: bool
35
(35, 7)


## Consulta 1 — Adelie de más de 4000 g

In [30]:
print(mascara_adelies.sum())   # 35

# 35 de 152 Adelie. Menos de uno de cada cuatro: los Adelie son de los pequeños.
print(df[mascara_adelies][["species", "island", "body_mass_g"]].head())

35
   species     island  body_mass_g
7   Adelie  Torgersen       4675.0
9   Adelie  Torgersen       4250.0
14  Adelie  Torgersen       4400.0
17  Adelie  Torgersen       4500.0
19  Adelie  Torgersen       4200.0


In [31]:
# OJO con el denominador. La pregunta era "¿qué proporción DE LOS ADELIE pasa de
# 4000 g?", así que primero recorto a Adelie y LUEGO calculo.
#
#   con &          -> mean() divide entre los 344 del dataset  -> ~10 %
#   recortando     -> mean() divide entre los 152 Adelie       -> ~23 %
#
# Las dos cuentan 35, pero solo la segunda responde la pregunta. .mean() siempre
# divide entre TODAS las filas de lo que tenga delante: manda qué le pongo delante.

adelie = df[df["species"] == "Adelie"]
print((adelie["body_mass_g"] > 4000).mean() * 100)

23.026315789473685


## Consulta 2 — De Biscoe o de Dream

Dos formas de escribir lo mismo. La segunda escala, la primera no.

In [32]:
# Con el operador |
mascara_islas = (df["island"] == "Biscoe") | (df["island"] == "Dream")
print(mascara_islas.sum())   # 292

# Con isin(): acepta cualquier colección iterable (lista, tupla, set, otra Serie)
print(df["island"].isin(["Biscoe", "Dream"]).sum())   # 292

# Prefiero isin porque LA LISTA PUEDE VENIR DE FUERA: de un CSV, de otra columna,
# de lo que elija el usuario. Con | tendría que reescribir el código cada vez que
# cambie la lista. Es la misma lección de la Fase 0: nada de valores a mano.
#
# Aviso para datos reales: isin distingue mayúsculas. "biscoe" != "Biscoe", y
# " Biscoe " con espacio tampoco cuela. Por eso lo primero con texto es normalizarlo.

292
292


## Consulta 3 — Los que NO son Gentoo

Lo importante de esta consulta no es el `~`. Es la **validación**.

In [33]:
# El ~ va DELANTE del paréntesis que envuelve la condición entera. Primero se
# resuelve la comparación, después se invierte el resultado.
mascara_no_gentoo = ~(df["species"] == "Gentoo")
print(mascara_no_gentoo.sum())   # 220

print(df.groupby("species").size())

# VALIDACIÓN: 152 + 68 = 220, y 220 + 124 = 344. Cuadra.
#
# Esta comprobación es la parte que importa. Un filtro mal escrito casi nunca da
# error: devuelve menos filas de las que debería y el número parece razonable.
# La primera vez que corrí esto olvidé el ~ y salió 124 (justo los Gentoo).
# Sin la tabla al lado me lo habría creído.
#
# Regla: cuando filtres, comprueba que las partes suman el total.

220
species
Adelie       152
Chinstrap     68
Gentoo       124
dtype: int64


In [34]:
# Nota sobre ~: solo tiene sentido sobre máscaras booleanas.
#   ~"Gentoo"            -> TypeError, avisa
#   ~df["body_mass_g"]   -> NO avisa: hace una operación de bits y devuelve
#                           -(n+1), o sea pesos negativos. Silencioso y roto.
# Úsalo siempre pegado a un paréntesis que contenga una comparación.

## Consulta 4 — Hembras de más de 5000 g

Aquí lo interesante no es el código: es lo que dice el resultado.

In [35]:
mascara_hembras = (df["sex"] == "Female") & (df["body_mass_g"] > 5000)

print(mascara_hembras.sum())   # 5

# Pido solo las columnas que voy a mirar. Con todas, Pandas parte la tabla en dos
# bloques y marca el corte con una barra invertida al final de la línea.
print(df[mascara_hembras]
      .sort_values("body_mass_g", ascending=False)
      [["species", "island", "body_mass_g"]])

# sort_values() ordena de MENOR a mayor por defecto: ascending=False lo invierte.

5
    species  island  body_mass_g
293  Gentoo  Biscoe       5200.0
342  Gentoo  Biscoe       5200.0
254  Gentoo  Biscoe       5150.0
268  Gentoo  Biscoe       5100.0
252  Gentoo  Biscoe       5050.0


### Conclusión: el filtro no hace lo que parece

**Las 5 son Gentoo, y las 5 de Biscoe.** No hay ni una Adelie ni una Chinstrap.

No es casualidad. Las Gentoo hembra pesan 4680 g de media frente a los 3369 g de
las Adelie: **1300 g de diferencia entre especies**. Así que un umbral de 5000 g
no está separando por sexo, está separando por especie sin que yo se lo pidiera.

Y "las hembras grandes están todas en Biscoe" suena a hallazgo y no lo es: los
Gentoo **solo** viven en Biscoe. La isla no explica nada, es un reflejo de la especie.

Es un error de interpretación clásico: una variable que no he pedido se cuela y
explica el resultado. Ningún código lo detecta. Solo mirar la salida y preguntarse
por qué sale así.

In [36]:
# Comprobación de que Gentoo solo vive en Biscoe
print(df.groupby(["species", "island"]).size())

species    island   
Adelie     Biscoe        44
           Dream         56
           Torgersen     52
Chinstrap  Dream         68
Gentoo     Biscoe       124
dtype: int64


## Lo que me llevo

1. **Filtrar son dos pasos**: la condición fabrica la máscara, los corchetes la aplican.
2. **`&`, `|`, `~`, `isin`** en vez de `and`, `or`, `not`, `in` — y siempre con paréntesis.
3. **El denominador depende de qué haya delante de `.mean()`**. Recortar primero
   cambia la pregunta que estoy respondiendo.
4. **Validar que las partes suman el total.** Un filtro mal escrito no da error,
   da menos filas y un número creíble.
5. **Mirar el resultado, no solo que el código funcione.** El filtro de la consulta 4
   funcionaba perfectamente y respondía a otra pregunta.

### Pendiente para la sesión 3

Hoy he pedido filas (`df[mascara]`) y columnas (`df[["a", "b"]]`) por separado.
¿Cómo se piden **las dos a la vez**? Ahí entran `.loc` y `.iloc`.

Pista de por qué hay dos: al filtrar, Pandas **conserva el índice original**
(los números 293, 342, 254... de la consulta 4, que no van seguidos). Así que
"la tercera fila" es ambiguo: ¿la de posición 3, o la de índice 3?